# CHAPTER 05 · 컴퓨터 비전과 시퀀스 모델 - 실습

- **실습 1** YOLO로 사진 속 물체 찾기 — `5_3.ipynb` · 셀 5개 · 학습 안 함
- **실습 2** U-Net으로 이미지 분할하기 — `5_4.ipynb` · 셀 5개 · 5 에폭 학습
- **실습 3** 어텐션으로 영어 → 한국어 번역하기 — `5_6.ipynb` · 셀 6개 · 30 에폭 학습 (100 권장)

| | 실습 1 YOLO | 실습 2 U-Net | 실습 3 어텐션 번역 |
|---|---|---|---|
| 문제 | 물체 위치 찾기 (탐지) | 픽셀 단위로 구분 (분할) | 문장 → 문장 (번역) |
| 입력 | 사진 1장 | 64×64 흑백 도형 그림 | 영어 문장 → 번호 8개 |
| 출력 | 상자 · 클래스 · 점수 | 64×64 확률 지도 (0~1) | 한국어 단어 번호 |
| 모델 | YOLOv8n — 불러와서 사용 | UNetMini — 직접 만듦 | Seq2Seq + 어텐션 — 직접 만듦 |
| 학습 | 안 함 | 5 에폭 | 30 에폭 (100 권장) |
| 손실 함수 | — | BCELoss | CrossEntropyLoss |

학습하는 두 실습의 학습 루프는 4장과 똑같은 5줄입니다:
`optimizer.zero_grad()` → `model(...)` → `criterion(...)` → `loss.backward()` → `optimizer.step()`

## 개념 · 객체 탐지 = 무엇이 + 어디에

- **이미지 인식** — 사진 전체에 이름 1개
- **객체 탐지** — 물체마다 상자·이름·점수 (실습 1)
- **의미 분할** — 픽셀마다 이름 (실습 2는 도형/배경 두 가지)
- **개체별 분할** — 같은 종류도 한 마리씩 따로 구분

**YOLO = You Only Look Once**
사진을 한 번만 훑어 물체를 모두 찾음 → 빠름. COCO(80종)로 이미 학습된 모델을 쓰기만 함

## [프로그램 5-1] YOLOv8 모델 불러오기

In [ ]:
import torch

# YOLO 설치
!pip install ultralytics
from ultralytics import YOLO

# 장치 선택
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 사전 학습된 모델 로드
model = YOLO("yolov8n.pt").to(device)
# model2 = YOLO("yolov8s.pt").to(device)  # 과제 4: yolov8n → yolov8s로 바꿔서 결과·속도(ms) 비교

### 셀 1 설명 — 모델 준비

- `!pip install ultralytics` — `!`로 시작하면 파이썬이 아니라 터미널 명령. 코랩에 없는 라이브러리 설치 (런타임 연결이 끊겨 새로 연결되면 다시 실행)
- `ultralytics` — YOLOv8을 만든 회사의 라이브러리. 여기서 `YOLO` 클래스 하나만 가져옴
- `import torch` — 장치가 있는지 확인하려고 가져옴
- `torch.cuda.is_available()` — GPU가 있으면 `"cuda"`, 없으면 `"cpu"`. 코랩 기본 런타임은 cpu, GPU 런타임이면 cuda
- `YOLO("yolov8n.pt")` — 파일이 없으면 자동 다운로드 (약 6MB). COCO 80종으로 학습이 끝난 가중치
- `.to(device)` — 모델을 고른 장치로 옮기기 (cpu면 그대로)

| yolov8?.pt | n (nano) | s | m | l | x |
|---|---|---|---|---|---|
| 파라미터 수 | 3.2M | 11.2M | 25.9M | 43.7M | 68.2M |

## [프로그램 5-2] 테스트 이미지 불러오기



In [76]:
#from PIL import Image

#img = Image.open("sample.jpg")     # RGB 이미지

#COCO 이미지 예시를 사용
from PIL import Image
from io import BytesIO
import requests

#url = "http://farm9.staticflickr.com/8261/8702818172_05105ccf66_z.jpg"
#url = "https://cdn.sisajournal-e.com/news/photo/202402/400433_200475_5137.jpg" # 위쪽뷰
#url = "https://www.paxetv.com/news/photo/202609/282762_258396_3649.jpg"
url = "https://cdn.thetrippick.com/news/photo/202507/420_1597_571.jpg"

resp = requests.get(url, timeout=10)
resp.raise_for_status()  # HTTP 오류면 예외

img = Image.open(BytesIO(resp.content)).convert("RGB")

### 셀 2 설명 — 테스트 이미지 준비

흐름: 주소(url) → 내려받기(`requests.get`) → 파일 내용(`resp.content`) → 파일처럼(`BytesIO`) → 이미지로(`Image.open`) → 3채널로(`.convert("RGB")`)

- `timeout=10` — 10초 안에 응답이 없으면 포기
- `raise_for_status()` — 404 같은 HTTP 오류면 여기서 바로 멈춤
- `.convert("RGB")` — 흑백이나 투명(RGBA) 사진도 3채널로 통일

**내 사진으로 바꾸기 (1~3번 줄 = 내 파일을 여는 버전)**
1. 1·3번 줄의 `#`을 지우고, 6~15번 줄은 `#`으로 막기
2. 코랩 왼쪽 파일 창에 사진 올리기
3. 파일 우클릭 → 경로 복사 → 3번 줄 따옴표 안에 붙여넣기 (예: `img = Image.open("/content/sample.jpg")`)

**자주 하는 실수**: `requests.get()`에 파일 경로를 넣으면 `MissingSchema` 에러. `requests`는 `http://` 주소 전용, 파일은 `Image.open()`으로.
주소가 사라지면 `HTTPError`, 인터넷·방화벽으로 접속이 막히면 `ConnectionError`·`Timeout` → 다른 주소나 내 사진 사용

## [프로그램 5-3] 모델 추론 실행하기


In [ ]:
results = model.predict(img, device=device)
results2 = model.predict(img, classes=[8], device=device, conf=0.5)
# results3 = model2.predict(img, device=device)  # 과제 4: yolov8s 모델로 같은 사진 추론해서 비교
# results4 = model2.predict(img, classes=[8], device=device, conf=0.5)  # 과제 4: yolov8s + classes 필터

### 셀 3 설명 — predict 한 줄 안에서 일어나는 일

① 전처리(640에 맞춤) → ② 순전파(후보 수천 개) → ③ NMS(거르기) → ④ 좌표 복원(원본 크기로)

- **① 전처리**: 긴 변을 640에 맞추고, 짧은 변은 32의 배수가 되게 회색으로 채움(레터박스) + 0~1 텐서로. 4장의 ToTensor를 알아서 해 줌
- **② 순전파**: 칸마다 상자 후보를 한 번에 예측 — 640×640이면 8,400칸, 448×640이면 5,880칸
- **③ NMS**: 점수 0.25 미만은 버리고, 같은 클래스끼리 많이 겹치면 점수가 가장 높은 것 하나만 남김
- **④ 좌표 복원**: 640 기준 좌표를 원본 사진 크기로 되돌림

출력 모양 예시 (다른 예시 사진, 가로 640×세로 427을 CPU로 돌린 결과 · 셀 2 사진이면 물체 목록·크기·시간이 다름):
```
0: 448x640 4 persons, 1 bus, 80.7ms
Speed: 3.9ms preprocess, 80.7ms inference, 4.9ms postprocess per image at shape (1, 3, 448, 640)
```
- `448x640` — 레터박스로 맞춘 입력 크기 (세로×가로 순서: 세로 427 → 448)
- preprocess·inference·postprocess = 위 ①, ②, ③④에 걸린 시간

**predict 옵션 — 결과를 바꾸는 설정값**

| 옵션 = 기본값 | 뜻 | 바꾸면 |
|---|---|---|
| conf=0.25 | 점수가 이보다 낮은 상자는 버림 | 올리면 확실한 것만 남음 |
| iou=0.7 | 같은 클래스 상자가 이만큼 이상 겹치면 하나만 남김 | 내리면 겹친 상자가 더 많이 지워짐 |
| classes=None | 찾을 클래스 번호 목록 (None = 80종 전부) | [0]이면 사람만 |
| imgsz=640 | 입력 크기 | 키우면 작은 물체를 잘 찾지만 느림 |

바꿔 보기 — 옵션은 괄호 안에 이름=값으로 추가 (한 번에 하나씩): `results = model.predict(img, device=device, conf=0.5)`

- 반환값 `results`는 리스트 — 사진 1장을 넣었으니 원소 1개 → `results[0]`

## [프로그램 5-4] 탐지 결과를 텍스트로 확인하기

In [ ]:
print("Results (전체 클래스):")
for r in results:
    for box in r.boxes:
        cls_id = int(box.cls)
        score = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()   # 경계 상자 좌표
        print(f"class={cls_id}, score={score:.3f}, box=({x1:.1f},{y1:.1f})-({x2:.1f},{y2:.1f})")
        # print(f"class={model.names[cls_id]}, score={score:.3f}, box=({x1:.1f},{y1:.1f})-({x2:.1f},{y2:.1f})")  # 과제 5: 번호 대신 이름

print("Results2 (classes=[8], boat만):")
for r in results2:
    for box in r.boxes:
        cls_id = int(box.cls)
        score = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()   # 경계 상자 좌표
        print(f"class={cls_id}, score={score:.3f}, box=({x1:.1f},{y1:.1f})-({x2:.1f},{y2:.1f})")

# 과제 4: yolov8s(model2) 결과 확인 — 셀 1의 model2, 셀 3의 results3/results4를 먼저 주석 해제해야 함
# print("Results3 (YOLOv8s):")
# for r in results3:
#     for box in r.boxes:
#         cls_id = int(box.cls)
#         score = float(box.conf)
#         x1, y1, x2, y2 = box.xyxy[0].tolist()
#         print(f"model={model2.names[cls_id]}[{cls_id}], score={score:.3f}, box=({x1:.1f},{y1:.1f})-({x2:.1f},{y2:.1f})")
#
# print("Results4 (YOLOv8s, classes=[8]):")
# for r in results4:
#     for box in r.boxes:
#         cls_id = int(box.cls)
#         score = float(box.conf)
#         x1, y1, x2, y2 = box.xyxy[0].tolist()
#         print(f"model={model2.names[cls_id]}[{cls_id}], score={score:.3f}, box=({x1:.1f},{y1:.1f})-({x2:.1f},{y2:.1f})")

### 셀 4 설명 — results 안에 든 것

흐름: 리스트(사진 수만큼) `results` → 사진 1장의 결과 `r` → 찾은 상자 N개 `r.boxes` → 상자 1개 `box`

| 속성 | 뜻 | 출력 예 |
|---|---|---|
| box.cls | 클래스 번호 (0 person · 2 car · 5 bus …) | tensor([0.]) |
| box.conf | 점수 0 ~ 1 | tensor([0.8700]) |
| box.xyxy | 상자 좌표 — 왼쪽 위 (x1, y1), 오른쪽 아래 (x2, y2) | tensor([[x1, y1, x2, y2]]) |

- `int()` · `float()` — 값이 1개뿐인 텐서를 파이썬 숫자로 바꾸기
- `box.xyxy[0]` — (1, 4)에서 한 줄 꺼내 (4,) → `.tolist()` → 변수 4개로 풀기
- `{score:.3f}` 소수 셋째 자리, `{x1:.1f}` 첫째 자리까지
- 번호 대신 이름: `model.names[cls_id]` → `'person'`

출력 모양 (숫자는 예시): `class=0, score=0.884, box=(529.4,154.1)-(639.7,343.9)`

## [프로그램 5-5] 탐지 결과를 시각화하기


In [79]:
results[0].save("detected.jpg")      # 탐지 결과 이미지 저장
results[0].show()                    # 주피터/콜랩 환경에서 시각화

results2[0].save("detected.jpg")      # 탐지 결과 이미지 저장
results2[0].show()                    # 주피터/콜랩 환경에서 시각화


### 셀 5 설명 — 상자를 그린 사진 저장 · 보기

- `save("detected.jpg")` — 상자 · 이름 · 점수를 그린 사진을 파일로 저장 (왼쪽 파일 창에서 확인)
- `show()` — 코랩에선 셀 아래에 바로 그림 표시 (내 PC 파이썬에선 새 창)
- `plot()`은 그림을 BGR 순서로 돌려줌 → `[:, :, ::-1]`로 뒤집어야 색이 정상

그림이 안 보이는 환경이면:
```python
import matplotlib.pyplot as plt
plt.imshow(results[0].plot()[:, :, ::-1])
plt.show()
```

## 과제 · 직접 바꿔 보기 — 실습 1

1. **내 사진으로**: 셀 2의 1·3번 줄 주석을 풀고 6~15번 줄은 `#`으로 막은 뒤, 3번 줄에 사진 경로 넣기
2. **predict에 conf=0.5 추가**: 상자 수가 어떻게 줄어드나요?
3. **predict에 classes=[0] 추가**: 사람만 찾기. [2](car), [5](bus)도 해 보기
4. **yolov8n.pt → yolov8s.pt**: 셀 1의 12번 줄. 결과와 속도(ms)를 비교해 보기
5. **model.names[cls_id]**: 셀 4 출력에 번호 대신 이름이 나오게 바꾸기

- 한 번에 하나씩만 바꾸세요. 두 개를 같이 바꾸면 무엇 때문에 달라졌는지 알 수 없습니다.
- 바꾼 셀부터 셀 5까지 순서대로 다시 실행합니다 (과제 4는 셀 1부터).

## 참고 · 실습 중 에러가 나면

| 에러 · 증상 | 원인 | 해결 |
|---|---|---|
| ModuleNotFoundError: No module named 'ultralytics' | 설치 안 됨 (새 런타임 포함) | `!pip install ultralytics` |
| MissingSchema | 경로를 requests.get에 넣음 | 파일은 `Image.open("경로")` |
| HTTPError (404 등) | 이미지 주소가 사라짐 | 다른 주소나 내 사진 사용 |
| ConnectionError · Timeout | 인터넷·방화벽으로 접속이 막힘 | 내 사진 사용 (셀 2의 1·3번 줄) |
| 사람이 파랗게 나옴 | plot()은 BGR 순서 | `[:, :, ::-1]`로 뒤집기 |